In [ ]:
import os
import pandas as pd
import numpy as np
import re
import requests

In [ ]:
# Define user
user = os.getlogin()

# Set file paths
path_sp  = os.path.join('C:\\Users', user, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_git = os.path.join('C:\\Users', user, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_config = os.path.join(path_git, 'Python Code', 'DOF', 'aa_config')
print(path_sp)

In [ ]:
## Outline
# Set fake user agent to avoid 403 error
# Set URL of table, starting here https://dof.ca.gov/forecasting/demographics/
# Request import of excel workbook with URL
# Convert request content to pandas dataframe
# Drop missings, remove state totals
# Reshape data to long format, clean date field, reshape back to wide
# Clean column names
# Repeat for data from 2000-2010, 2010-2020, 2020-2024
# Stack all data together
# Subset to counties as needed


# i got lucky with finding this user agent on stackoverflow, not sure why it works
headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/114.0'} 


## Import data
# 2000-2010
url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E4_2000-2010_Report_Final_EOC_000.xlsx"
request = requests.get(url, headers = headers)
request = request.content
df_2000w = pd.read_excel(request, sheet_name = 1, skiprows = 3, engine='openpyxl')
df_2000w = df_2000w.dropna()
df_2000w = df_2000w[df_2000w['COUNTY'] != 'State Total']
df_2000 = pd.melt(df_2000w
                    , id_vars = ['COUNTY']
                    , var_name = 'Date'
                    , value_name = 'Total'
                 )
df_2000['Date'] = pd.to_datetime(df_2000['Date'])
df_2000w = df_2000.pivot_table(index = ['COUNTY']
                                   , columns = 'Date'
                                   , values = 'Total').reset_index()
df_2000w.columns = [re.sub(" 00:00:00", "", str(col)) for col in df_2000w.columns]

# 2010-2020
url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-4_2010-2020-Internet-Version.xlsx"
request = requests.get(url, headers = headers)
request = request.content
df_2010w = pd.read_excel(request, sheet_name = 1, skiprows = 1, engine='openpyxl')
df_2010w = df_2010w[df_2010w['COUNTY'] != 'State Total']
df_2010w = df_2010w.dropna()
df_2010 = pd.melt(df_2010w
                    , id_vars = ['COUNTY']
                    , var_name = 'Date'
                    , value_name = 'Total'
                 )
df_2010['Date'] = pd.to_datetime(df_2010['Date'])
df_2010w = df_2010.pivot_table(index = ['COUNTY']
                                   , columns = 'Date'
                                   , values = 'Total').reset_index()
df_2010w.columns = [re.sub(" 00:00:00", "", str(col)) for col in df_2010w.columns]

# 2020-2024
url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-4_2024_InternetVersion.xlsx"
request = requests.get(url, headers = headers)
request = request.content
df_2020w = pd.read_excel(request, sheet_name = 1, skiprows = 2, engine='openpyxl')
df_2020w = df_2020w[df_2020w['COUNTY'] != 'State Total']
df_2020w = df_2020w.dropna()
df_2020 = pd.melt(df_2020w
                    , id_vars = ['COUNTY']
                    , var_name = 'Date'
                    , value_name = 'Total'
                 )
df_2020['Date'] = pd.to_datetime(df_2020['Date'])
df_2020w = df_2020.pivot_table(index = ['COUNTY']
                                   , columns = 'Date'
                                   , values = 'Total').reset_index()
df_2020w.columns = [re.sub(" 00:00:00", "", str(col)) for col in df_2020w.columns]

## Combine data

# wide
df_dof2 = df_2000w.merge(df_2010w, on = 'COUNTY', how = 'left')
df_dof2 = df_dof2 .merge(df_2020w, on = 'COUNTY', how = 'left')

# long
df_dof = pd.concat([df_2000, df_2010, df_2020])
df_dof = df_dof.sort_values(['COUNTY', 'Date'])

## Subset to counties as needed
df_dof  = df_dof [df_dof ['COUNTY'].isin(counties)]
df_dof2 = df_dof2[df_dof2['COUNTY'].isin(counties)]

In [ ]:
df_dof

In [ ]:
df_dof2

Code graveyard

In [ ]:
# header = {
#   "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.75 Safari/537.36",
#   "X-Requested-With": "XMLHttpRequest"
# }

# r = requests.get(site, headers=header)
# r.text

In [ ]:
# import pandas as pd
# import requests

# # Check the end of the url -->                                                                             HERE --v
# url = 'https://<myOrg>.sharepoint.com/:x:/s/x-taulukot/Ec0R1y3l7sdGsP92csSO-mgBI8WCN153LfEMvzKMSg1Zzg?e=6NS5Qh&download=1'
# headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/114.0'}

# resp = requests.get(url, headers=headers)
# df = pd.read_excel(resp.content, engine='openpyxl')